In [2]:
import ast
import numpy as np
import pandas as pd
credits = pd.read_csv("credits.csv")
# Corrected function with np.nan and a safeguard for missing/malformed data
def get_director(x):
    # Convert string representation of list to an actual Python list
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return np.nan
    # Loop through the actual list of dictionaries
    if isinstance(x, list):
        for i in x:
            if i.get("job") == "Director":
                return i.get("name")
    return np.nan  # Fixed from np.na
# Apply the function to create the 'director' column
credits["director"] = credits["crew"].apply(get_director)
credits= credits.drop(columns=['crew','cast'])
credits.head()


,id,director
0,862,John Lasseter
1,8844,Joe Johnston
2,15602,Howard Deutch
3,31357,Forest Whitaker
4,11862,Charles Shyer


In [3]:
metadata = pd.read_csv('movies_metadata.csv')
# Create your new 'genre' column from your original column (assuming it's called 'genres')
def get_first_genre(x):
    # Convert string representation into a real Python list
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return np.nan

    # Target the list directly
    if isinstance(x, list) and len(x) > 0:
        first_item = x[0]  # Grab the very first dictionary in the list
        if isinstance(first_item, dict) and "name" in first_item:
            return first_item["name"]
    return np.nan
# Create your new 'genre' column from your original 'genres' column
metadata["genre"] = metadata["genres"].apply(get_first_genre)
metadata1 = metadata[['id','title','overview','genre']]
metadata1.isnull().sum()

/tmp/ipykernel_23266/2215872852.py:1: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv('movies_metadata.csv')


id             0
title          6
overview     954
genre       2442
dtype: int64

In [4]:
rating = pd.read_csv('ratings_small.csv')
rating.head()
rating.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [5]:
# 1. Clean the 'id' column in metadata1 by forcing it to numeric and dropping bad rows
metadata1["id"] = pd.to_numeric(metadata1["id"], errors="coerce")
metadata1 = metadata1.dropna(subset=["id"])
# 2. Convert both ID columns to the exact same integer type
metadata1["id"] = metadata1["id"].astype(int)
credits["id"] = credits["id"].astype(int)
# 3. Perform the merge smoothly
metadata1 = metadata1.merge(credits, on="id")
metadata1.head()

,id,title,overview,genre,director
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation,John Lasseter
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,Adventure,Joe Johnston
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance,Howard Deutch
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy,Forest Whitaker
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,Charles Shyer


In [6]:
metadata1.info()

<class 'pandas.DataFrame'>
RangeIndex: 45538 entries, 0 to 45537
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        45538 non-null  int64
 1   title     45535 non-null  str  
 2   overview  44584 non-null  str  
 3   genre     43096 non-null  str  
 4   director  44651 non-null  str  
dtypes: int64(1), str(4)
memory usage: 17.1 MB


In [7]:
# 1. Ensure movieId in ratings is converted to integer
rating["movieId"] = pd.to_numeric(rating["movieId"], errors="coerce")
rating = rating.dropna(subset=["movieId"])
rating["movieId"] = rating["movieId"].astype(int)
# 2. Merge ratings into your existing metadata1 DataFrame
metadata1 = metadata1.merge(rating, left_on="id", right_on="movieId")
# 3. Optional: Drop the duplicate 'movieId' column to keep the DataFrame clean
metadata1 = metadata1.drop(columns=["movieId"])
metadata1 = metadata1.rename(columns={"id": "movieId"})
metadata1.head()

,movieId,title,overview,genre,director,userId,rating,timestamp
0,949,Heat,"Obsessive master thief, Neil McCauley leads a ...",Action,Michael Mann,23,3.5,1148721092
1,949,Heat,"Obsessive master thief, Neil McCauley leads a ...",Action,Michael Mann,102,4.0,956598942
2,949,Heat,"Obsessive master thief, Neil McCauley leads a ...",Action,Michael Mann,232,2.0,955092697
3,949,Heat,"Obsessive master thief, Neil McCauley leads a ...",Action,Michael Mann,242,5.0,956688825
4,949,Heat,"Obsessive master thief, Neil McCauley leads a ...",Action,Michael Mann,263,3.0,1117846575


In [8]:
metadata1.genre.value_counts()

genre
Drama              14855
Comedy              7885
Action              5632
Adventure           3139
Crime               2916
Horror              1967
Thriller            1563
Fantasy             1399
Documentary          995
Romance              983
Science Fiction      890
Animation            494
Mystery              468
Music                456
Western              422
Family               230
War                  225
TV Movie             148
History               72
Foreign               65
Name: count, dtype: int64

In [9]:
metadata1.to_csv("movies.csv", index=False)